# Adhoc: количество операций ТЭ и СБП за май–июль 2026

На выходе 6 цифр: `trx_cnt` торгового эквайринга и СБП за май, июнь, июль 2026.

**ТЭ** — как `05_transaction_metrics`: транзакция `c_trx_class = SA` + договор `acq_class = SA`.

**СБП** — по договору, как отчёт 214: `agreements.acq_class = QP`.
В `scd1_trx` класса `QP` нет (probe это подтвердил). Поэтому СБП не режем по `c_trx_class`, `S01` и RSHB — только месяц, не удалено, не статус `R`, связка `trx_acq` → договор QP.


In [ ]:
import sys
from pathlib import Path

import pandas as pd
from rail_connectors.connection import connect

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.float_format', lambda x: f'{x:,.0f}'.replace(',', ' '))

LAKE_USER = 'Shestopalov-VYur'
LAKE_PASSWORD = None
for cand in [
    Path.cwd(),
    Path.cwd() / 'sources' / 'sql',
    Path('/home/jovyan/documents/Equaring'),
    Path('/home/jovyan/documents/Equaring/sources/sql'),
]:
    if (cand / 'connection_secrets.py').exists():
        if str(cand) not in sys.path:
            sys.path.insert(0, str(cand))
        from connection_secrets import LAKE_USER, LAKE_PASSWORD
        break

user_params = {'user_name': LAKE_USER}
if LAKE_PASSWORD:
    user_params['password'] = LAKE_PASSWORD

imp = connect(
    to='IMPALA',
    extra_options={'db': 'sandbox_ai'},
    driver_args={'tez.queue.name': 'ai'},
    kerberos={
        'keytab_path': '/home/jovyan/test_requests/tech.keytab',
        'use_credentials': True,
        'update_keytab': True,
    },
    user_params=user_params,
)
imp._init_connection()
print('imp ready, user =', LAKE_USER)

In [ ]:
PERIOD_START = '2026-05-01'
PERIOD_END_EXCL = '2026-08-01'
MEM_LIMIT = '16g'

# rows_cnt == trx_cnt — это нормально: в зерне class×type нет дублей n_trx.
# Важно: в trx нет класса QP. QP — это acq_class договора (10548 шт.).
sql_qp_funnel = f"""
select
  substr(cast(trunc(to_date(cast(t.d_trx_orig as timestamp)), 'MM') as string), 1, 7) as report_month,
  upper(trim(cast(t.c_trx_class as string))) as trx_class,
  t.c_trx_type as trx_type,
  count(distinct t.n_trx) as trx_cnt
from ods_alpha.scd1_agreements a
join ods_alpha.scd1_trx_acq acq
  on cast(acq.n_agr as string) = cast(a.n_agr as string)
join ods_alpha.scd1_trx t
  on cast(t.n_trx as string) = cast(acq.n_trx as string)
where upper(trim(cast(a.acq_class as string))) = 'QP'
  and coalesce(a.ods_deleted_flg, '0') <> '1'
  and coalesce(t.ods_deleted_flg, '0') <> '1'
  and cast(t.d_trx_orig as timestamp) >= cast('{PERIOD_START}' as timestamp)
  and cast(t.d_trx_orig as timestamp) < cast('{PERIOD_END_EXCL}' as timestamp)
group by 1, 2, 3
order by trx_cnt desc
limit 40
"""

with imp:
    imp.execute(f'set MEM_LIMIT={MEM_LIMIT}')
    qp_funnel = imp.fetch(sql_qp_funnel)

print('Операции на договорах QP (СБП) за май–июль: какой trx_class / trx_type')
display(qp_funnel)

In [ ]:
MONTHS = [
    ('2026-05-01', '2026-05-31'),
    ('2026-06-01', '2026-06-30'),
    ('2026-07-01', '2026-07-31'),
]
PERIOD_START = '2026-05-01'
PERIOD_END_EXCL = '2026-08-01'
MEM_LIMIT = '16g'

months_union_sql = '\n  union all\n'.join(
    f"  select cast('{start}' as date) as month_start, cast('{end}' as date) as month_end"
    for start, end in MONTHS
)

sql = f"""
with months as (
{months_union_sql}
),
fiid_rshb as (
  select distinct cast(fa.c_fiid as string) as c_fiid
  from ods_alpha.scd1_base24_fiids fa
  where coalesce(cast(fa.c_fiid_grp as string), 'UNKNOWN') = 'RSHB'
),
agr_terms_p as (
  select distinct
    m.month_start,
    cast(t.n_agr as string) as n_agr
  from months m
  join ods_alpha.scd1_agr_terms t
    on cast(t.d_valid_from as date) <= m.month_end
   and (t.d_valid_to is null or cast(t.d_valid_to as date) > m.month_start)
  where upper(trim(cast(t.cf_ter_type as string))) = 'P'
    and coalesce(t.ods_deleted_flg, '0') <> '1'
),
agr_sa as (
  select distinct
    m.month_start,
    'Торговый эквайринг' as product,
    cast(a.n_agr as string) as n_agr
  from months m
  join ods_alpha.scd1_agreements a
    on cast(a.d_valid_from as date) <= m.month_end
   and (a.d_valid_to is null or cast(a.d_valid_to as date) >= m.month_start)
  join ods_alpha.scd1_companies c
    on c.n_cmp = a.n_cmp_client
  join agr_terms_p tp
    on tp.n_agr = cast(a.n_agr as string)
   and tp.month_start = m.month_start
  where upper(trim(cast(a.acq_class as string))) = 'SA'
    and coalesce(a.ods_deleted_flg, '0') <> '1'
    and coalesce(c.ods_deleted_flg, '0') <> '1'
    and c.c_inn is not null
),
agr_qp as (
  select distinct
    m.month_start,
    'СБП' as product,
    cast(a.n_agr as string) as n_agr
  from months m
  join ods_alpha.scd1_agreements a
    on cast(a.d_valid_from as date) <= m.month_end
   and (a.d_valid_to is null or cast(a.d_valid_to as date) >= m.month_start)
  where upper(trim(cast(a.acq_class as string))) = 'QP'
    and coalesce(a.ods_deleted_flg, '0') <> '1'
),
trx_te as (
  select
    cast(trunc(to_date(cast(t.d_trx_orig as timestamp)), 'MM') as date) as month_start,
    cast(t.n_trx as string) as n_trx
  from ods_alpha.scd1_trx t
  join fiid_rshb fr
    on fr.c_fiid = cast(t.c_fiid_acq as string)
  where cast(t.d_trx_orig as timestamp) >= cast('{PERIOD_START}' as timestamp)
    and cast(t.d_trx_orig as timestamp) < cast('{PERIOD_END_EXCL}' as timestamp)
    and coalesce(t.ods_deleted_flg, '0') <> '1'
    and coalesce(t.cf_trx_stat, '') <> 'R'
    and t.c_trx_type = 'S01'
    and upper(trim(cast(t.c_trx_class as string))) = 'SA'
    and t.c_nter is not null
),
trx_sbp as (
  select
    cast(trunc(to_date(cast(t.d_trx_orig as timestamp)), 'MM') as date) as month_start,
    cast(t.n_trx as string) as n_trx
  from ods_alpha.scd1_trx t
  where cast(t.d_trx_orig as timestamp) >= cast('{PERIOD_START}' as timestamp)
    and cast(t.d_trx_orig as timestamp) < cast('{PERIOD_END_EXCL}' as timestamp)
    and coalesce(t.ods_deleted_flg, '0') <> '1'
    and coalesce(t.cf_trx_stat, '') <> 'R'
),
ta_sa as (
  select
    agr.month_start,
    agr.product,
    tb.n_trx
  from ods_alpha.scd1_trx_acq acq
  join trx_te tb
    on tb.n_trx = cast(acq.n_trx as string)
  join agr_sa agr
    on agr.n_agr = cast(acq.n_agr as string)
   and agr.month_start = tb.month_start
),
ta_qp as (
  select
    agr.month_start,
    agr.product,
    tb.n_trx
  from ods_alpha.scd1_trx_acq acq
  join trx_sbp tb
    on tb.n_trx = cast(acq.n_trx as string)
  join agr_qp agr
    on agr.n_agr = cast(acq.n_agr as string)
   and agr.month_start = tb.month_start
),
ta as (
  select month_start, product, n_trx from ta_sa
  union all
  select month_start, product, n_trx from ta_qp
)
select
  substr(cast(month_start as string), 1, 7) as report_month,
  product,
  count(distinct n_trx) as trx_cnt
from ta
group by 1, 2
order by 1, 2
"""

with imp:
    imp.execute(f'set MEM_LIMIT={MEM_LIMIT}')
    raw = imp.fetch(sql)

if raw is None or raw.empty:
    raw = pd.DataFrame(columns=['report_month', 'product', 'trx_cnt'])

raw['trx_cnt'] = pd.to_numeric(raw['trx_cnt'], errors='coerce').fillna(0).astype('int64')
raw['report_month'] = raw['report_month'].astype(str).str[:7]

result = (
    raw.pivot_table(index='product', columns='report_month', values='trx_cnt', aggfunc='sum')
    .reindex(index=['Торговый эквайринг', 'СБП'], columns=['2026-05', '2026-06', '2026-07'])
    .fillna(0)
    .astype('int64')
)

display(result)
print()
for product in result.index:
    for month in result.columns:
        print(f'{product} {month}: {int(result.loc[product, month]):,}'.replace(',', ' '))